# Week 2: Point-in-Time Feature Engineering & Online-Offline Parity

This notebook explores the 22 engineered features, demonstrates mathematical formulation for balance accounting discrepancies, explains stateful destination profile tracking, and validates online-offline feature parity.

In [ ]:
import pandas as pd
import numpy as np
import json
from src.config import PROCESSED_DATA_DIR, MODELS_DIR
from src.features import FEATURE_NAMES, CustomerStateCache, extract_features

# 1. Load engineered feature metadata
with open(MODELS_DIR / 'feature_meta.json') as f:
    meta = json.load(f)

print(f"Total features engineered: {meta['feature_count']}")
print("Feature list:", meta['feature_names'])

## 1. Balance Discrepancy Formulations

In non-fraudulent banking transactions:
$$\text{newbalanceOrig} = \text{oldbalanceOrg} - \text{amount} \implies \text{errorBalanceOrig} = \text{newbalanceOrig} + \text{amount} - \text{oldbalanceOrg} = 0$$

$$\text{newbalanceDest} = \text{oldbalanceDest} + \text{amount} \implies \text{errorBalanceDest} = \text{oldbalanceDest} + \text{amount} - \text{newbalanceDest} = 0$$

In fraud, these equations frequently break because:
1. The entire account balance is drained (`newbalanceOrig = 0.0`) regardless of whether the amount matched the full balance.
2. The destination account is a mule that immediately forwards or liquidates cash (`newbalanceDest = 0.0`).

In [ ]:
# Inspect discrepancy distributions on validation set
val_feats = pd.read_parquet(PROCESSED_DATA_DIR / 'val_features.parquet')
print("Balance Error Orig stats (Fraud vs Non-Fraud):")
print(val_feats.groupby('isFraud')['errorBalanceOrig'].describe())

print("\nBalance Error Dest stats (Fraud vs Non-Fraud):")
print(val_feats.groupby('isFraud')['errorBalanceDest'].describe())

## 2. Stateful Point-in-Time Aggregations

To prevent **temporal data leakage**, when computing rolling metrics for `nameDest`:
- For any row at index $i$, the rolling transaction count and amount sum incorporate **only** transactions at indices $< i$.
- The historical state at the end of the Train partition (steps 1–550) is carried forward as the initial state for the Validation partition (steps 551–650).

In [ ]:
print("Destination rolling features summary in Validation:")
val_feats[['dest_prev_tx_count', 'dest_prev_amount_sum', 'dest_prev_amount_mean']].describe()

## 3. Online-Offline Parity & Latency Benchmark

In production (FastAPI), incoming transactions must be scored in under 100 ms.
We verify that `CustomerStateCache.extract_online_vector()` produces an identical feature vector to the offline batch pipeline and runs in **< 10 microseconds**.

In [ ]:
import time

sample_tx = {
    'step': 600,
    'type': 'TRANSFER',
    'amount': 150000.0,
    'nameOrig': 'C1234567',
    'oldbalanceOrg': 150000.0,
    'newbalanceOrig': 0.0,
    'nameDest': 'C9876543',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
}

cache = CustomerStateCache()

# Warmup
for _ in range(50):
    _ = cache.extract_online_vector(sample_tx)

# Measure latency over 10,000 iterations
start = time.perf_counter()
n_iter = 10000
for _ in range(n_iter):
    vec = cache.extract_online_vector(sample_tx)
elapsed_us = ((time.perf_counter() - start) / n_iter) * 1_000_000

print(f"Measured Online Feature Extraction Latency: {elapsed_us:.2f} microseconds (< 0.01 ms)")
print(f"Vector shape: {vec.shape} (Matches 22 canonical features)")